In [ ]:
import geopandas as gpd
import pandas as pd

import cafo_iowa.db.session as s
from cafo_iowa.estimate.estimate import load_and_process_facilities

import cafo_iowa.db.models as m
import cafo_iowa.db.session as s

pd.set_option("display.max_columns", None)

import datetime

today = datetime.datetime.now().strftime("%Y%m%d")

au_plot_limit = 2500

In [ ]:
facilities = load_and_process_facilities()
# restrict to swine facilities
facilities = facilities[facilities.animal_cat_combined == "swine"]

In [ ]:
# get all empty swine facilities
empty_fac = facilities[(facilities.barn_count == 0)]
empty_fac_permits = empty_fac.permit_ids.explode().tolist()

# get all tiles that intersect with empty facilities
query = f"""
WITH empty_permits AS (
    SELECT id, facilityid
    FROM processed.permits
    WHERE id IN ({','.join(map(str, empty_fac_permits))})
),
parcels_for_empty_permits AS (
    SELECT p.id as parcel_id, p.geometry
    FROM processed.parcels p
    JOIN processed.permit_parcels pp ON p.id = pp.parcel_id
    JOIN empty_permits ep ON pp.permit_id = ep.id
),
tiles_with_parcels AS (
    SELECT DISTINCT n.id as naip_qt_id
    FROM processed.naip21_qt n
    JOIN parcels_for_empty_permits p ON ST_Intersects(n.geometry, p.geometry)
),
tiles_with_permits AS (
    SELECT 
        naip_qt_id, 
        ARRAY_AGG(DISTINCT id) AS permit_ids
    FROM processed.permits
    WHERE id IN ({','.join(map(str, empty_fac_permits))})
    GROUP BY naip_qt_id
)
SELECT 
    COALESCE(tp.naip_qt_id, tpar.naip_qt_id) as naip_qt_id,
    COALESCE(tp.permit_ids, ARRAY[]::integer[]) as permit_ids
FROM tiles_with_permits tp
FULL OUTER JOIN tiles_with_parcels tpar ON tp.naip_qt_id = tpar.naip_qt_id;
"""

empty_facilities_tile_ids = pd.read_sql(query, s.get_engine())
empty_facilities_tile_ids.to_csv(
    "data/relabeling/empty_facilities_tile_ids.csv", index=False
)
print(empty_facilities_tile_ids.shape)
print(len(empty_fac_permits))

In [ ]:
# get tiles for underestimated swine permits
# where estimated_animal_units < reported_animal_units /2 and estimated_animal_units > 0
underestimated_fac = facilities[
    (facilities.estimated_animal_units <= facilities.reported_animal_units / 2)
    & (facilities.estimated_animal_units > 0)
    & (facilities.swine_cat_combined.isin(["swine_wean", "swine_grow"]))
]
underestimated_fac_permits = underestimated_fac.permit_ids.explode().tolist()

# get tiles for underestimated swine facilities
query = f"""
WITH underestimated_permits AS (
    SELECT id, facilityid
    FROM processed.permits
    WHERE id IN ({','.join(map(str, underestimated_fac_permits))})
),
parcels_for_underestimated_permits AS (
    SELECT p.id as parcel_id, p.geometry
    FROM processed.parcels p
    JOIN processed.permit_parcels pp ON p.id = pp.parcel_id
    JOIN underestimated_permits ep ON pp.permit_id = ep.id
),
tiles_with_parcels AS (
    SELECT DISTINCT n.id as naip_qt_id
    FROM processed.naip21_qt n
    JOIN parcels_for_underestimated_permits p ON ST_Intersects(n.geometry, p.geometry)
),
tiles_with_permits AS (
    SELECT 
        naip_qt_id, 
        ARRAY_AGG(DISTINCT id) AS permit_ids
    FROM processed.permits
    WHERE id IN ({','.join(map(str, empty_fac_permits))})
    GROUP BY naip_qt_id
)
SELECT 
    COALESCE(tp.naip_qt_id, tpar.naip_qt_id) as naip_qt_id,
    COALESCE(tp.permit_ids, ARRAY[]::integer[]) as permit_ids
FROM tiles_with_permits tp
FULL OUTER JOIN tiles_with_parcels tpar ON tp.naip_qt_id = tpar.naip_qt_id;"""

underestimated_facilities_tile_ids = pd.read_sql(query, s.get_engine())
underestimated_facilities_tile_ids.to_csv(
    "data/relabeling/underestimated_facilities_tile_ids.csv", index=False
)
print(underestimated_facilities_tile_ids.shape)
print(len(underestimated_fac_permits))

In [ ]:
# get tiles for overestimated swine permits
# where estimated_animal_units >= reported_animal_units /2 and estimated_animal_units > 0
overestimated_fac = facilities[
    (facilities.estimated_animal_units >= facilities.reported_animal_units * 2)
    & (facilities.estimated_animal_units > 0)
    & (facilities.swine_cat_combined.isin(["swine_wean", "swine_grow"]))
]
overestimated_fac_permits = overestimated_fac.permit_ids.explode().tolist()

# get tiles for underestimated swine facilities
query = f"""
WITH overestimated_permits AS (
    SELECT id, facilityid
    FROM processed.permits
    WHERE id IN ({','.join(map(str, overestimated_fac_permits))})
),
parcels_for_overestimated_permits AS (
    SELECT p.id as parcel_id, p.geometry
    FROM processed.parcels p
    JOIN processed.permit_parcels pp ON p.id = pp.parcel_id
    JOIN overestimated_permits ep ON pp.permit_id = ep.id
),
tiles_with_parcels AS (
    SELECT DISTINCT n.id as naip_qt_id
    FROM processed.naip21_qt n
    JOIN parcels_for_overestimated_permits p ON ST_Intersects(n.geometry, p.geometry)
),
tiles_with_permits AS (
    SELECT 
        naip_qt_id, 
        ARRAY_AGG(DISTINCT id) AS permit_ids
    FROM processed.permits
    WHERE id IN ({','.join(map(str, empty_fac_permits))})
    GROUP BY naip_qt_id
)
SELECT 
    COALESCE(tp.naip_qt_id, tpar.naip_qt_id) as naip_qt_id,
    COALESCE(tp.permit_ids, ARRAY[]::integer[]) as permit_ids
FROM tiles_with_permits tp
FULL OUTER JOIN tiles_with_parcels tpar ON tp.naip_qt_id = tpar.naip_qt_id;"""

overestimated_facilities_tile_ids = pd.read_sql(query, s.get_engine())
overestimated_facilities_tile_ids.to_csv(
    "data/relabeling/overestimated_facilities_tile_ids.csv", index=False
)
print(overestimated_facilities_tile_ids.shape)
print(len(overestimated_fac_permits))

In [ ]:
# get all tile ids for unlabeled tiles that are within 100 m of swine permits
query = """
WITH labeled_tile_ids AS (
    -- Extract all tile IDs from label_batches
    SELECT DISTINCT jsonb_array_elements_text(naip_qt_ids) AS tile_id
    FROM processed.label_batches
    WHERE naip_qt_ids IS NOT NULL
),
unlabeled_tiles AS (
    -- Get all tiles from naip21_qt tha t are not in labeled_tile_ids
    SELECT n.id, n.geometry, n.is_urban, n.urban_area
    FROM processed.naip21_qt n
    LEFT JOIN labeled_tile_ids lti ON n.id = lti.tile_id
    WHERE lti.tile_id IS NULL
),
tiles_near_swine_permits AS (
    -- Find unlabeled tiles that are near swine permits
    SELECT 
        ut.id, 
        ut.geometry, 
        ut.is_urban, 
        ut.urban_area,
        COUNT(DISTINCT p.id) AS nearby_swine_permits,
        SUM(p.swine_animal_units) AS total_swine_animal_units
    FROM unlabeled_tiles ut
    JOIN processed.permits p ON ST_DWithin(p.geometry, ut.geometry, 100)  -- 100 meters
    WHERE p.swine_animal_units > 0  -- Only swine permits
    GROUP BY ut.id, ut.geometry, ut.is_urban, ut.urban_area
)
SELECT 
	id as naip_qt_id
FROM tiles_near_swine_permits
"""

unlabeled_swine_tile_ids = pd.read_sql(query, s.get_engine())

# load unlabeled tiles from manual review
unlabeled_tiles_manual = pd.read_csv("data/manual_review/tiles_to_label_{today}.csv")

# merge with unlabeled tiles from adjacent permits
unlabeled_tiles_manual = unlabeled_tiles_manual.rename(
    columns={"tile_id": "naip_qt_id"}
)

unlabeled_swine_tile_ids = pd.concat([unlabeled_swine_tile_ids, unlabeled_tiles_manual])
unlabeled_swine_tile_ids.drop_duplicates(subset=["naip_qt_id"], inplace=True)

print(unlabeled_swine_tile_ids.shape)

unlabeled_swine_tile_ids["naip_qt_id"].to_csv(
    "data/relabeling/unlabeled_adjacent_tiles_near_permits.csv",
    index=False,
)